# Export Gold datasets for Tableau

This notebook reads the five published Gold Parquet datasets from MinIO and exports one local CSV per dataset for Tableau Public/Community Edition. It is designed for this project's small, seeded dataset; `toPandas()` should not be used for large production-scale datasets.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

from src.spark_minio import get_spark_session

load_dotenv()
bucket_name = os.environ["MINIO_BUCKET"]
spark = get_spark_session("sanlam-export-gold-for-tableau")
spark.sparkContext.setLogLevel("WARN")

GOLD_ROOT = f"s3a://{bucket_name}/gold"
EXPORT_DIRECTORY = Path.cwd().parent / "tableau_exports"
EXPORT_DIRECTORY.mkdir(exist_ok=True)

In [ ]:
GOLD_TABLES = [
    "customer_360",
    "policy_performance",
    "claims_performance",
    "premium_collection",
    "policy_status_history",
]

gold = {
    table_name: spark.read.parquet(f"{GOLD_ROOT}/{table_name}/")
    for table_name in GOLD_TABLES
}

for table_name, dataframe in gold.items():
    print(f"Loaded {table_name}: {dataframe.count()} rows")

In [ ]:
# Convert each small Gold DataFrame to one CSV file.
# index=False prevents pandas from adding an unwanted numbered index column.
for table_name, dataframe in gold.items():
    output_file = EXPORT_DIRECTORY / f"{table_name}.csv"
    dataframe.toPandas().to_csv(output_file, index=False, encoding="utf-8")
    print(f"Exported {table_name} -> {output_file}")

print("Tableau CSV export completed successfully.")

## Tableau next steps

In Tableau, select **Connect → Text file**, then choose one CSV at a time from `tableau_exports/`. Start each dashboard from its matching Gold CSV; the datasets are already reporting-ready and normally do not need joins in Tableau.